In [1]:
import pandas as pd

In [29]:
transacciones = pd.read_parquet('transacciones.parquet', engine='fastparquet')
clientes = pd.read_parquet('clientes.parquet', engine='fastparquet')
productos = pd.read_parquet('productos.parquet', engine='fastparquet')

In [30]:
start_date = '2025-01-01'
end_date = '2025-01-05'

# Generar un rango de fechas (lista de objetos datetime.date o pd.Timestamp)
fechas = pd.date_range(start=start_date, end=end_date, freq='D')
num_dias = len(fechas)

print(f"Rango de fechas: {fechas[0].date()} a {fechas[-1].date()} ({num_dias} días)")

# 2.2 Extraer las columnas clave para el cruce
clientes_ids = clientes['customer_id'].tolist()
productos_ids = productos['product_id'].tolist()

# 2.3 Generar el Producto Cartesiano
# La función product de itertools es la más eficiente para generar todas las combinaciones.
# La combinación debe ser: (customer_id, product_id, purchase_date)
print("\nGenerando producto cartesiano (Combinando Clientes x Productos x Fechas)...")

# Se usa list comprehension sobre el resultado de product() para crear la lista de tuplas
combinaciones = list(product(clientes_ids, productos_ids, fechas))

# 2.4 Crear el DataFrame final
df_combinado = pd.DataFrame(combinaciones, columns=['customer_id', 'product_id', 'purchase_date'])

# Convertir la columna de fechas a tipo date para coincidir con el formato deseado
df_combinado['purchase_date'] = df_combinado['purchase_date'].dt.date

# ----------------------------------------------------------------------
# 3. Verificación de Resultados
# ----------------------------------------------------------------------
filas_esperadas = len(clientes) * len(productos) * num_dias
filas_obtenidas = len(df_combinado)

print("\n--- Resultado ---")
print(f"Filas esperadas (C x P x D): {len(clientes)} x {len(productos)} x {num_dias} = {filas_esperadas}")
print(f"Filas obtenidas en el DataFrame: {filas_obtenidas}")
print("\nPrimeras 10 filas del DataFrame Combinado:")
print(df_combinado.head(10).to_markdown(index=False))

print("\nÚltimas 5 filas del DataFrame Combinado:")
print(df_combinado.tail().to_markdown(index=False))

# Opcional: Para verificar que todas las fechas están representadas
print("\nRecuento de filas por día:")
print(df_combinado['purchase_date'].value_counts().sort_index().to_markdown())

Rango de fechas: 2025-01-01 a 2025-01-05 (5 días)

Generando producto cartesiano (Combinando Clientes x Productos x Fechas)...

--- Resultado ---
Filas esperadas (C x P x D): 1569 x 971 x 5 = 7617495
Filas obtenidas en el DataFrame: 7617495

Primeras 10 filas del DataFrame Combinado:
|   customer_id |   product_id | purchase_date   |
|--------------:|-------------:|:----------------|
|        256017 |        34092 | 2025-01-01      |
|        256017 |        34092 | 2025-01-02      |
|        256017 |        34092 | 2025-01-03      |
|        256017 |        34092 | 2025-01-04      |
|        256017 |        34092 | 2025-01-05      |
|        256017 |        57290 | 2025-01-01      |
|        256017 |        57290 | 2025-01-02      |
|        256017 |        57290 | 2025-01-03      |
|        256017 |        57290 | 2025-01-04      |
|        256017 |        57290 | 2025-01-05      |

Últimas 5 filas del DataFrame Combinado:
|   customer_id |   product_id | purchase_date   |
|---------

In [51]:
import pandas as pd

# ⚠️ Asegúrate de que tu DataFrame esté cargado.
# df_transacciones = ...

# --- 1. Definir la columna de fecha y el rango ---
# Asumo que la columna de fecha se llama 'purchase_date'
COLUMNA_FECHA = 'purchase_date' 
FECHA_INICIO = '2024-12-23'
FECHA_FIN = '2024-12-29'

# --- 2. Asegurar el tipo de dato datetime (si es necesario) ---
# Si la columna ya es datetime, este paso se puede omitir o ignorar.
transacciones[COLUMNA_FECHA] = pd.to_datetime(transacciones[COLUMNA_FECHA], errors='coerce')


# --- 3. Aplicar el filtro booleano ---
# El filtro selecciona las filas donde la fecha es MAYOR O IGUAL a la fecha de inicio
# Y MENOR O IGUAL a la fecha de fin.
filtro_semana = (
    (transacciones[COLUMNA_FECHA] >= FECHA_INICIO) & 
    (transacciones[COLUMNA_FECHA] <= FECHA_FIN)
)

# Aplicar el filtro para crear el nuevo DataFrame
df_semana_filtrado = transacciones[filtro_semana].copy()

# Mostrar un resumen del resultado
print(f"Filas originales: {len(transacciones):,}")
print(f"Filas después del filtro (23-29 Dic 2024): {len(df_semana_filtrado):,}")
print("\n✅ DataFrame filtrado completado.")

Filas originales: 254,936
Filas después del filtro (23-29 Dic 2024): 8,216

✅ DataFrame filtrado completado.


In [52]:
df_semana_filtrado = df_semana_filtrado[["customer_id","product_id","purchase_date"]]

In [34]:
import joblib
import pandas as pd
from typing import List, Dict, Any

# ===============================================
# 0. Lógica de Preprocesamiento Requerida
# ===============================================

def preprocess_data_for_prediction(df: pd.DataFrame) -> pd.DataFrame:
    """
    Replica el preprocesamiento necesario para que los datos sean compatibles
    con el modelo, especialmente la conversión de fechas.
    """
    df_proc = df.copy() # Trabaja en una copia para evitar SettingWithCopyWarning
    
    # 1. Renombrar (basado en tu código anterior)
    df_proc = df_proc.rename(columns={
        'client_id': 'customer_id',      
        'product_id': 'product_id',       
        'date': 'purchase_date'        
    })
    
    # Define las constantes que usaste
    WEEK_COL = "week" 
    
    # 2. Convertir y crear la columna WEEK_COL
    df_proc["purchase_date"] = pd.to_datetime(df_proc["purchase_date"], format='%Y-%m-%d', errors="coerce")
    # Nota: Tu código usa .dt.to_period("W"), lo replicamos aquí.
    df_proc[WEEK_COL] = df_proc["purchase_date"].dt.to_period("W").apply(lambda r: r.start_time)
    
    # 3. Conversión de DATETIME a INT64 (La clave para resolver el error)
    datetime_cols = [c for c in df_proc.columns if c != WEEK_COL and pd.api.types.is_datetime64_any_dtype(df_proc[c])]
    for col in datetime_cols:
        # Convertir el tipo datetime a su representación numérica (timestamp int64)
        df_proc[col] = df_proc[col].view("int64") 
        
    # 4. Seleccionar las features finales en el orden correcto
    # Tu código indica que las features finales son 'customer_id', 'product_id', 'purchase_date'
    orden = ['customer_id', 'product_id', 'purchase_date'] 
    
    # Aseguramos que solo pasamos las columnas que el modelo espera.
    df_final = df_proc[orden]
    
    print(f"Columnas finales preprocesadas: {list(df_final.columns)}")
    print(f"Tipos de datos finales (d_types): {df_final.dtypes.to_dict()}")
    
    return df_final


# ===============================================
# 1. Carga del modelo (Mantenemos la carga ya exitosa)
# ===============================================
ruta_relativa = r"C:\Users\JaviZ\MDS7202\Proyecto\Entrega 2\app\backend\model_artifacts\trained_model.joblib"

try:
    modelo_cargado = joblib.load(ruta_relativa)
    print("✅ Modelo cargado exitosamente.")
except Exception as e:
    # Manejo de error si la ruta falla
    print(f"⚠️ Error al cargar el modelo: {e}")
    # Si la carga falla, el resto del script fallará si no definimos el modelo
    # Para la prueba, asumiremos que se cargó, pero en un entorno real, ¡detente aquí!


# ===============================================
# 2. Aplicación del Preprocesamiento
# ===============================================

# ⚠️ Importante: df_combinado debe estar disponible aquí.
# Asumiendo que df_combinado ya existe y contiene 'customer_id', 'product_id', 'purchase_date'
# (o los nombres originales 'client_id', 'product_id', 'date' si el preprocesamiento los renombra)

# ➡️ Aplicar la función de preprocesamiento al DataFrame de entrada
datos_prediccion_procesados = preprocess_data_for_prediction(df_combinado)

print("\nDatos de entrada (DataFrame) listos para la predicción:")
print(datos_prediccion_procesados.head())


# ===============================================
# 3. Utilización del modelo para la predicción
# ===============================================
try:
    # Pasamos el DataFrame PROCESADO, donde 'purchase_date' ahora es int64
    prediccion = modelo_cargado.predict(datos_prediccion_procesados)
    probabilidades = modelo_cargado.predict_proba(datos_prediccion_procesados)
    
    # ===============================================
    # 4. Resultado
    # ===============================================
    print("\n===============================================")
    print(f"Resultado de la Predicción (Primera fila): {prediccion[0]}")
    print(f"Probabilidades (Primera fila): {probabilidades[0]}")
    print(f"Total de predicciones generadas: {len(prediccion)}")
    print("===============================================")

except Exception as e:
    print(f"\n❌ Falló la predicción: {e}")
    print("Asegúrate de que la cantidad y el orden de las columnas en 'datos_prediccion_procesados' coincidan con las features de entrenamiento.")

✅ Modelo cargado exitosamente.


C:\Users\JaviZ\AppData\Local\Temp\ipykernel_21800\669697825.py:35: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  df_proc[col] = df_proc[col].view("int64")


Columnas finales preprocesadas: ['customer_id', 'product_id', 'purchase_date']
Tipos de datos finales (d_types): {'customer_id': dtype('int64'), 'product_id': dtype('int64'), 'purchase_date': dtype('int64')}

Datos de entrada (DataFrame) listos para la predicción:
   customer_id  product_id        purchase_date
0       256017       34092  1735689600000000000
1       256017       34092  1735776000000000000
2       256017       34092  1735862400000000000
3       256017       34092  1735948800000000000
4       256017       34092  1736035200000000000

Resultado de la Predicción (Primera fila): 1
Probabilidades (Primera fila): [0.21279836 0.78720164]
Total de predicciones generadas: 7617495


In [54]:
import joblib
import pandas as pd
from typing import List, Dict, Any

# ===============================================
# 0. Lógica de Preprocesamiento Requerida
# ===============================================

def preprocess_data_for_prediction(df: pd.DataFrame) -> pd.DataFrame:
    """
    Replica el preprocesamiento necesario para que los datos sean compatibles
    con el modelo, especialmente la conversión de fechas.
    """
    df_proc = df.copy() # Trabaja en una copia para evitar SettingWithCopyWarning
    
    # 1. Renombrar (basado en tu código anterior)
    df_proc = df_proc.rename(columns={
        'client_id': 'customer_id',      
        'product_id': 'product_id',       
        'date': 'purchase_date'        
    })
    
    # Define las constantes que usaste
    WEEK_COL = "week" 
    
    # 2. Convertir y crear la columna WEEK_COL
    df_proc["purchase_date"] = pd.to_datetime(df_proc["purchase_date"], format='%Y-%m-%d', errors="coerce")
    # Nota: Tu código usa .dt.to_period("W"), lo replicamos aquí.
    df_proc[WEEK_COL] = df_proc["purchase_date"].dt.to_period("W").apply(lambda r: r.start_time)
    
    # 3. Conversión de DATETIME a INT64 (La clave para resolver el error)
    datetime_cols = [c for c in df_proc.columns if c != WEEK_COL and pd.api.types.is_datetime64_any_dtype(df_proc[c])]
    for col in datetime_cols:
        # Convertir el tipo datetime a su representación numérica (timestamp int64)
        df_proc[col] = df_proc[col].view("int64") 
        
    # 4. Seleccionar las features finales en el orden correcto
    # Tu código indica que las features finales son 'customer_id', 'product_id', 'purchase_date'
    orden = ['customer_id', 'product_id', 'purchase_date'] 
    
    # Aseguramos que solo pasamos las columnas que el modelo espera.
    df_final = df_proc[orden]
    
    print(f"Columnas finales preprocesadas: {list(df_final.columns)}")
    print(f"Tipos de datos finales (d_types): {df_final.dtypes.to_dict()}")
    
    return df_final


# ===============================================
# 1. Carga del modelo (Mantenemos la carga ya exitosa)
# ===============================================
ruta_relativa = r"C:\Users\JaviZ\MDS7202\Proyecto\Entrega 2\app\backend\model_artifacts\trained_model.joblib"

try:
    modelo_cargado = joblib.load(ruta_relativa)
    print("✅ Modelo cargado exitosamente.")
except Exception as e:
    # Manejo de error si la ruta falla
    print(f"⚠️ Error al cargar el modelo: {e}")
    # Si la carga falla, el resto del script fallará si no definimos el modelo
    # Para la prueba, asumiremos que se cargó, pero en un entorno real, ¡detente aquí!


# ===============================================
# 2. Aplicación del Preprocesamiento
# ===============================================

# ⚠️ Importante: df_combinado debe estar disponible aquí.
# Asumiendo que df_combinado ya existe y contiene 'customer_id', 'product_id', 'purchase_date'
# (o los nombres originales 'client_id', 'product_id', 'date' si el preprocesamiento los renombra)

# ➡️ Aplicar la función de preprocesamiento al DataFrame de entrada
datos_prediccion_procesados = preprocess_data_for_prediction(df_semana_filtrado)

print("\nDatos de entrada (DataFrame) listos para la predicción:")
print(datos_prediccion_procesados.head())


# ===============================================
# 3. Utilización del modelo para la predicción
# ===============================================
try:
    # Pasamos el DataFrame PROCESADO, donde 'purchase_date' ahora es int64
    prediccion = modelo_cargado.predict(datos_prediccion_procesados)
    probabilidades = modelo_cargado.predict_proba(datos_prediccion_procesados)
    
    # ===============================================
    # 4. Resultado
    # ===============================================
    print("\n===============================================")
    print(f"Resultado de la Predicción (Primera fila): {prediccion[0]}")
    print(f"Probabilidades (Primera fila): {probabilidades[0]}")
    print(f"Total de predicciones generadas: {len(prediccion)}")
    print("===============================================")

except Exception as e:
    print(f"\n❌ Falló la predicción: {e}")
    print("Asegúrate de que la cantidad y el orden de las columnas en 'datos_prediccion_procesados' coincidan con las features de entrenamiento.")

✅ Modelo cargado exitosamente.
Columnas finales preprocesadas: ['customer_id', 'product_id', 'purchase_date']
Tipos de datos finales (d_types): {'customer_id': dtype('int64'), 'product_id': dtype('int64'), 'purchase_date': dtype('int64')}

Datos de entrada (DataFrame) listos para la predicción:
        customer_id  product_id        purchase_date
164835       187806       39186  1734912000000000000
166418       187806       56714  1734912000000000000
172410       187806        1304  1734912000000000000
173502       187806        3382  1734912000000000000
174231       187806       11262  1734912000000000000

Resultado de la Predicción (Primera fila): 1
Probabilidades (Primera fila): [0.07452229 0.92547771]
Total de predicciones generadas: 8216


C:\Users\JaviZ\AppData\Local\Temp\ipykernel_21800\1186149057.py:35: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  df_proc[col] = df_proc[col].view("int64")


In [35]:
prediccion

array([1, 1, 1, ..., 1, 1, 1], dtype=int64)

In [55]:
import numpy as np

# --- Asumiendo que esta variable ya existe después de ejecutar la Sección 3 ---
# prediccion = modelo_cargado.predict(datos_prediccion_procesados) 
# ---

# 1. Convierte la lista/array de predicciones a un array de NumPy (si aún no lo es)
prediccion_np = np.array(prediccion)

# 2. Cuenta los valores
# Cuenta los ceros (0)
conteo_ceros = np.count_nonzero(prediccion_np == 0)

# Cuenta los unos (1)
conteo_unos = np.count_nonzero(prediccion_np == 1)

# 3. Muestra los resultados

print("## 📊 Conteo de Predicciones (0 y 1) ##")
print(f"Total de predicciones: {len(prediccion_np)}")
print("---")
print(f"Predicciones de Clase 0 (No Compra/Negativo): **{conteo_ceros:,}**")
print(f"Predicciones de Clase 1 (Compra/Positivo): **{conteo_unos:,}**")
print("===================================")

## 📊 Conteo de Predicciones (0 y 1) ##
Total de predicciones: 8216
---
Predicciones de Clase 0 (No Compra/Negativo): **0**
Predicciones de Clase 1 (Compra/Positivo): **8,216**


# Hasta Aqui llegan las predicciones correctas, lo de abajo es simplemente para generar csv y empaquetarlos en zip

In [42]:
import pandas as pd
import numpy as np

# --- Asumiendo que df_combinado está disponible y cargado ---

# 1. Crear el nuevo DataFrame con solo las columnas requeridas
df_id = df_combinado[['customer_id', 'product_id']].copy()
df_id = df_id.drop_duplicates()
print(f"Número de filas original: {len(df_id):,}")

# 2. Eliminar el 25% de las filas de forma aleatoria

# Calcular el número de filas a eliminar (25%)
filas_a_eliminar = int(len(df_id) * 0)
print(f"Número de filas a eliminar (25%): {filas_a_eliminar:,}")

# Obtener los índices de las filas a eliminar de forma aleatoria
indices_a_eliminar = df_id.sample(n=filas_a_eliminar, random_state=42).index
# Usamos random_state=42 para asegurar que la eliminación sea reproducible

# Eliminar las filas
df_reducido = df_id.drop(indices_a_eliminar)

print(f"Número de filas final: {len(df_reducido):,}")
print(f"Columnas finales: {list(df_reducido.columns)}")

# 3. Guardar el DataFrame reducido en un archivo CSV

nombre_archivo_csv = "customer_product_sample.csv"

# Guardar en CSV:
# - header=False: Para no incluir los nombres de las columnas
# - index=False: Para no incluir el índice de Pandas
# - sep=',': Para asegurar que el separador sea la coma
df_reducido.to_csv(
    nombre_archivo_csv, 
    header=False, 
    index=False, 
    sep=','
)

print("\n✅ Proceso completado.")
print(f"Archivo guardado exitosamente: **{nombre_archivo_csv}**")

Número de filas original: 1,523,499
Número de filas a eliminar (25%): 0
Número de filas final: 1,523,499
Columnas finales: ['customer_id', 'product_id']

✅ Proceso completado.
Archivo guardado exitosamente: **customer_product_sample.csv**


In [2]:
transacciones = pd.read_parquet('transacciones.parquet', engine='fastparquet')

In [2]:
import pandas as pd

In [3]:
new_batch = pd.read_parquet('batch_t3.parquet', engine='fastparquet')

In [4]:
new_batch

,customer_id,product_id,order_id,purchase_date,items
0,186933,57414,1311205,2025-01-18,1.000000
1,186933,57804,1311205,2025-01-18,1.000000
2,186933,50274,1311205,2025-01-18,1.666667
3,186933,33886,1311205,2025-01-18,5.666667
4,186933,30500,1311205,2025-01-18,5.666667
...,...,...,...,...,...
6227,66843,1370,1300885,2025-01-16,2.333333
6228,154119,1372,1306648,2025-01-17,1.000000
6229,154812,42132,1306659,2025-01-17,1.000000
6230,171684,11262,1301748,2025-01-16,1.000000


In [5]:
df_semana_filtrado_c_p = new_batch[["customer_id","product_id"]]
df_semana_filtrado_sin_duplicados = df_semana_filtrado_c_p.drop_duplicates()

In [6]:
nombre_archivo_csv = "customer_product_sample.csv"

# Guardar en CSV:
# - header=False: Para no incluir los nombres de las columnas
# - index=False: Para no incluir el índice de Pandas
# - sep=',': Para asegurar que el separador sea la coma
df_semana_filtrado_sin_duplicados.to_csv(
    nombre_archivo_csv, 
    header=False, 
    index=False, 
    sep=','
)

print("\n✅ Proceso completado.")
print(f"Archivo guardado exitosamente: **{nombre_archivo_csv}**")


✅ Proceso completado.
Archivo guardado exitosamente: **customer_product_sample.csv**


In [7]:
import zipfile
import os

# --- 1. Definir los nombres de los archivos ---
nombre_archivo_csv = "customer_product_sample.csv"
nombre_archivo_zip = "datos_comprimidos.zip"

# Asumiendo que el archivo CSV ya existe en el directorio de trabajo.
if not os.path.exists(nombre_archivo_csv):
    print(f"❌ Error: No se encontró el archivo '{nombre_archivo_csv}'. Asegúrate de que existe.")
else:
    # 2. Crear el archivo ZIP y agregar el CSV
    try:
        # Abre el archivo ZIP en modo escritura ('w')
        with zipfile.ZipFile(nombre_archivo_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
            
            # Agrega el archivo CSV al ZIP
            # El segundo argumento (opcional) es el nombre que tendrá el archivo dentro del zip.
            zipf.write(nombre_archivo_csv, arcname=nombre_archivo_csv)
        
        print(f"✅ Archivo '{nombre_archivo_csv}' comprimido exitosamente en '{nombre_archivo_zip}'.")
        
    except Exception as e:
        print(f"⚠️ Ocurrió un error durante la compresión: {e}")

✅ Archivo 'customer_product_sample.csv' comprimido exitosamente en 'datos_comprimidos.zip'.
